In [1]:
import tskit
import pandas as pd
from lets_plot import *
LetsPlot.setup_html()
from collections import defaultdict
import numpy as np
from collections import Counter

chromosome = 16

In [2]:
originalTs = tskit.load("All_snps_with_outgroup_info_newAA/Chr" + str(chromosome) + ".trees")
remappedTs = tskit.load("Remapped_newNe/Chr" + str(chromosome) + ".trees")
chrSample = pd.read_table("All_snps_with_outgroup_info_newAA/Chr" + str(chromosome) + "_relate.sample")


# %%
meta = pd.read_csv("TsinferMetaData_Vcf.txt")
meta.head()
allMeta = pd.read_csv("/home/jobste/Data/DataDescriptor_SampleMeta.csv")
allMeta.head()
meta.loc[:, "Country"] = [allMeta.Country[allMeta.Run == x].to_list()[0] for x in meta.ID]



In [ ]:
# Add country as pop

sampleMeta = chrSample.merge(meta, how = "left", left_on = "ID_1", right_on = "ID")
sampleMeta = sampleMeta.iloc[1: , :]


complex_schema_pop = tskit.MetadataSchema({
    'codec': 'json',
    'additionalProperties': False,
    'properties': {'pop': {'description': 'Lineage',
                           'type': 'string'},
                   'subpop': {'description': 'Subspecies',
                           'type': 'string'},
                   'country': {'description': 'Country',
                           'type': 'string'}},
    'required': ['pop'],
    'type': 'object',
})

originalTables = originalTs.dump_tables()
remappedTables = remappedTs.dump_tables()
originalTables.populations.metadata_schema = complex_schema_pop
remappedTables.populations.metadata_schema = complex_schema_pop

# Add data to population table
popdata_column = [{"pop": pop, "subpop":subpop, "country": country } for pop,subpop,country in set(list(zip(meta.Pop, meta.SubPop, meta.Country)))]
encoded_popdata_column = [
    remappedTables.populations.metadata_schema.validate_and_encode_row(r) for r in popdata_column
]
md, md_offset = tskit.pack_bytes(encoded_popdata_column)
originalTables.populations.set_columns(metadata=md, metadata_offset=md_offset)
remappedTables.populations.set_columns(metadata=md, metadata_offset=md_offset)

popDict = {y: x for x,y in enumerate(set(list(zip(meta.Pop, meta.SubPop, meta.Country))))}
ind_nodes = list([x.id for x in remappedTs.individuals()])
ind_pops = [popDict[(sampleMeta.Pop.to_list()[x], sampleMeta.SubPop.to_list()[x], sampleMeta.Country.to_list()[x])] for x in ind_nodes]

In [3]:
# Add subpop as pop

sampleMeta = chrSample.merge(meta, how = "left", left_on = "ID_1", right_on = "ID")
sampleMeta = sampleMeta.iloc[1: , :]


complex_schema_pop = tskit.MetadataSchema({
    'codec': 'json',
    'additionalProperties': False,
    'properties': {'pop': {'description': 'Lineage',
                           'type': 'string'},
                   'subpop': {'description': 'Subspecies',
                           'type': 'string'}},
    'required': ['pop'],
    'type': 'object',
})

originalTables = originalTs.dump_tables()
remappedTables = remappedTs.dump_tables()
originalTables.populations.metadata_schema = complex_schema_pop
remappedTables.populations.metadata_schema = complex_schema_pop

# Add data to population table
popdata_column = [{"pop": pop, "subpop":subpop} for pop,subpop in set(list(zip(meta.Pop, meta.SubPop)))]
encoded_popdata_column = [
    remappedTables.populations.metadata_schema.validate_and_encode_row(r) for r in popdata_column
]
md, md_offset = tskit.pack_bytes(encoded_popdata_column)
originalTables.populations.set_columns(metadata=md, metadata_offset=md_offset)
remappedTables.populations.set_columns(metadata=md, metadata_offset=md_offset)

popDict = {y: x for x,y in enumerate(set(list(zip(meta.Pop, meta.SubPop))))}
ind_nodes = list([x.id for x in remappedTs.individuals()])
ind_pops = [popDict[(sampleMeta.Pop.to_list()[x], sampleMeta.SubPop.to_list()[x])] for x in ind_nodes]


In [ ]:
# Add lineage as pop

sampleMeta = chrSample.merge(meta, how = "left", left_on = "ID_1", right_on = "ID")
sampleMeta = sampleMeta.iloc[1: , :]


complex_schema_pop = tskit.MetadataSchema({
    'codec': 'json',
    'additionalProperties': False,
    'properties': {'pop': {'description': 'Lineage',
                           'type': 'string'}},
    'required': ['pop'],
    'type': 'object',
})

originalTables = originalTs.dump_tables()
remappedTables = remappedTs.dump_tables()
originalTables.populations.metadata_schema = complex_schema_pop
remappedTables.populations.metadata_schema = complex_schema_pop

# Add data to population table
popdata_column = [{"pop": pop} for pop in set(meta.Pop)]
encoded_popdata_column = [
    remappedTables.populations.metadata_schema.validate_and_encode_row(r) for r in popdata_column
]
md, md_offset = tskit.pack_bytes(encoded_popdata_column)
originalTables.populations.set_columns(metadata=md, metadata_offset=md_offset)
remappedTables.populations.set_columns(metadata=md, metadata_offset=md_offset)

popDict = {y: x for x,y in enumerate(set(meta.Pop))}
ind_nodes = list([x.id for x in remappedTs.individuals()])
ind_pops = [popDict[sampleMeta.Pop.to_list()[x]] for x in ind_nodes]


In [7]:
indNodePopO = originalTables.nodes.population
indNodePopO[ind_nodes] = ind_pops
indNodePopR = remappedTables.nodes.population
indNodePopR[ind_nodes] = ind_pops
originalTables.nodes.population = indNodePopO
remappedTables.nodes.population = indNodePopR
originalTs_meta = originalTables.tree_sequence()
remappedTs_meta = remappedTables.tree_sequence()

In [9]:
remappedTs_meta.num_populations

31

In [ ]:

mu = 3.4e-9
sfsBranch = pd.DataFrame()
sfsSite = pd.DataFrame()
for pop in range(originalTs_meta.num_populations):
    print(pop)
    tmpBranch =  mu * originalTs_meta.allele_frequency_spectrum(sample_sets = [originalTs_meta.samples(pop)], mode='branch', polarised = True)
    tmpBranchDF = pd.DataFrame({"Count": range(len(tmpBranch)), "Set": "Original","Freq": tmpBranch})
    tmpBranchDF.loc[:, "Pop"] = pop
    sfsBranch = pd.concat([sfsBranch, tmpBranchDF])
    tmpSite = originalTs_meta.allele_frequency_spectrum(sample_sets = [originalTs_meta.samples(pop)], mode='site', polarised = True)
    tmpSiteDF = pd.DataFrame({"Count": range(len(tmpSite)),"Set": "Original", "Freq": tmpSite})
    tmpSiteDF.loc[:, "Pop"] = pop
    sfsSite = pd.concat([sfsSite, tmpSiteDF])
    tmpBranch =  mu * remappedTs_meta.allele_frequency_spectrum(sample_sets = [remappedTs_meta.samples(pop)], mode='branch', polarised = True)
    tmpBranchDF = pd.DataFrame({"Count": range(len(tmpBranch)), "Set": "Remapped","Freq": tmpBranch})
    tmpBranchDF.loc[:, "Pop"] = pop
    sfsBranch = pd.concat([sfsBranch, tmpBranchDF])
    tmpSite = remappedTs_meta.allele_frequency_spectrum(sample_sets = [remappedTs_meta.samples(pop)], mode='site', polarised = True)
    tmpSiteDF = pd.DataFrame({"Count": range(len(tmpSite)),"Set": "Remapped", "Freq": tmpSite})
    tmpSiteDF.loc[:, "Pop"] = pop
    sfsSite = pd.concat([sfsSite, tmpSiteDF])
sfsSite.loc[:, "Method"] = "Site"
sfsBranch.loc[:, "Method"] = "Branch"

sfsDF = pd.concat([sfsSite, sfsBranch])
sfsDF.loc[:, "LogFreq"] = np.log(sfsDF.Freq)

In [ ]:
sfsDF.loc[:, "Set_Method"] = sfsDF.Set + sfsDF.Method

In [45]:
sfsDF.to_csv("Chr" + str(chromosome) + "_CountrySFS.csv", index = None)

In [48]:
sfsDF.head()

,Count,Set,Freq,Pop,Method,LogFreq,Set_Method
0,0,Original,0.025625,0,Site,-3.664202,OriginalSite
1,1,Original,0.000061,0,Site,-9.699770,OriginalSite
2,2,Original,0.000287,0,Site,-8.156587,OriginalSite
3,3,Original,0.000643,0,Site,-7.349362,OriginalSite
4,4,Original,0.001436,0,Site,-6.545800,OriginalSite


In [19]:
sfsDF = pd.read_csv("Chr" + str(chromosome) + "_CountrySFS.csv")
popDictTs = {x.id: x.metadata["subpop"] + "_" + x.metadata["country"] for x in remappedTs_meta.populations()}

In [20]:
sfsDF.loc[:, "PopName"] = [popDictTs[num] for num in sfsDF.Pop]

In [21]:
sfsDF.head()

,Count,Set,Freq,Pop,Method,LogFreq,Set_Method,PopName
0,0,Original,0.025625,0,Site,-3.664202,OriginalSite,Admixed_Sweden
1,1,Original,0.000061,0,Site,-9.699770,OriginalSite,Admixed_Sweden
2,2,Original,0.000287,0,Site,-8.156587,OriginalSite,Admixed_Sweden
3,3,Original,0.000643,0,Site,-7.349362,OriginalSite,Admixed_Sweden
4,4,Original,0.001436,0,Site,-6.545800,OriginalSite,Admixed_Sweden


In [ ]:
sfsDF.head()

In [22]:
ggplot(data = sfsDF) + geom_point(aes(x = "Count", y = "LogFreq", colour = "Set_Method"), stat="identity", position = "dodge") + \
facet_wrap(facets = "PopName", scales = "free_x")

In [11]:
# Try getting them from the VCF
import allel
vcf = allel.read_vcf("/home/jobste/Data/Remapped/Chr" + str(chromosome) + "_remapped.vcf.gz")

# %%
meta = pd.read_csv("/home/jobste/Data/TsinferMetaData_Vcf.txt", index_col = "ID")

# %%

meta.loc[:, "Index"] = range(len(meta))
s = list(vcf['samples'])
meta = meta.reindex(s)
meta = meta.set_index("Index")

gt = allel.GenotypeArray(vcf['calldata/GT'])

#ACS
# Pop
pops = set(meta.Pop)

popDict = {pop: meta[meta.Pop == pop].index for pop in pops}
acsPop = gt.count_alleles_subpops(popDict)


In [12]:
popDict = {pop: meta[meta.Pop == pop].index for pop in pops}
acsPop = gt.count_alleles_subpops(popDict)



In [13]:
sfs_scikit = pd.DataFrame()
for pop in acsPop.keys():
    tmp =  popSFS = allel.sfs(np.array([x[1] for x in acsPop[pop]]))
    tmpDF = pd.DataFrame({"Count": range(len(tmp)), "Set": "Remapped","Freq": tmp, "Pop": pop})
    sfs_scikit = pd.concat([sfs_scikit, tmpDF])

sfs_scikit.loc[:, "LogFreq"] = np.log(sfs_scikit.Freq)




In [14]:
ggplot(data = sfs_scikit) + geom_point(aes(x = "Count", y = "LogFreq"), stat="identity", position = "dodge") + \
facet_wrap(facets = "Pop", scales = "free_x")

In [1]:
import cyvcf2

In [38]:
vcfAll = cyvcf2.VCF("/home/jobste/Data/Chr16_AllInfo_AC.vcf.gz")
vcfA = cyvcf2.VCF("/home/jobste/Data/Chr16_AllInfo_AC_ALineage.vcf.gz")
#vcfAll = cyvcf2.VCF("/home/jobste/Data/Chr16_imputed_outgroupInfo_AC.vcf.gz")
#vcfA = cyvcf2.VCF("/home/jobste/Data/Chr16_imputed_outgroupInfo_AC_ALineage.vcf.gz")

In [29]:
ancestralList = []
ancestralTypeList = []
for variant in vcfAll:
    if variant.INFO["Prob"] >= 0.9:
        ancestral = variant.INFO["Major"]
        ancestralType = "Major"
    elif (variant.INFO["Prob"] < 0.1) & (variant.INFO["Prob"] != -99):
        al = [variant.REF, variant.ALT[0]]
        ancestral = al[al != variant.INFO["Major"]]
        ancestralType = "Minor"
    else:
        if variant.INFO["MajorOutgroup"] != '0':
            ancestral = variant.INFO["MajorOutgroup"]
            if ancestral == variant.INFO["Major"]:
                ancestralType = "MajorOutgroup_Major"
            else:
                ancestralType = "MajorOutgroup_Minor"
        else:
            ancestral = 0
            ancestralType = "0"

    ancestralList.append(ancestral)
    ancestralTypeList.append(ancestralType)


In [39]:
Counter(ancestralTypeList)

Counter({'Major': 240386,
         '0': 109926,
         'MajorOutgroup_Minor': 23884,
         'Minor': 6610,
         'MajorOutgroup_Major': 1528})

In [116]:

ancestralList = []
for variant in vcfAll:
    if variant.INFO["Prob"] >= 0.9:
        ancestral = variant.INFO["Major"]
    else:
        ancestral = 0
    ancestralList.append(ancestral)
            

In [40]:
num = 0
acs = []
for variant in vcfA:
    if variant.INFO["Major"] == ancestralList[num]:
        acs.append(variant.INFO["AC"])
    else:
        if ancestralList[num] == 0:
            acs.append("Skip")
        else:
            acs.append(variant.INFO["AN"] - variant.INFO["AC"])
    num += 1


In [41]:
Counter(acs)

Counter({0: 127494,
         'Skip': 109926,
         5: 18797,
         4: 17358,
         6: 10267,
         3: 8740,
         7: 7405,
         65: 6972,
         8: 4283,
         2: 4197,
         9: 3967,
         1: 3953,
         10: 3765,
         11: 3014,
         12: 2610,
         60: 2247,
         14: 2108,
         61: 2009,
         13: 2001,
         15: 1893,
         64: 1688,
         16: 1546,
         59: 1484,
         17: 1362,
         19: 1337,
         18: 1275,
         58: 1166,
         20: 1116,
         56: 1009,
         22: 993,
         55: 985,
         57: 970,
         62: 950,
         21: 936,
         23: 920,
         26: 884,
         24: 851,
         54: 847,
         53: 825,
         50: 806,
         25: 789,
         29: 756,
         52: 750,
         28: 749,
         51: 728,
         27: 699,
         30: 690,
         45: 687,
         49: 685,
         31: 647,
         33: 639,
         36: 628,
         42: 626,
         48: 622

In [42]:
from collections import Counter
acsDF = pd.DataFrame.from_dict(Counter(acs), orient = "index")
acsDF.columns = ["Freq"]
acsDF.loc[:, "Count"] = acsDF.index
acsDF.loc[:, "LogFreq"] = np.log(acsDF.Freq)
acsDF = acsDF[acsDF.Count != "Skip"]

In [43]:
ggplot(data = acsDF) + geom_point(aes(x = "Count", y= "LogFreq"))

In [37]:
acs = [x for x,y in zip(acs, ancestralTypeList) if y != "MajorOutgroup_Minor"]
from collections import Counter
acsDF = pd.DataFrame.from_dict(Counter(acs), orient = "index")
acsDF.columns = ["Freq"]
acsDF.loc[:, "Count"] = acsDF.index
acsDF.loc[:, "LogFreq"] = np.log(acsDF.Freq)
acsDF = acsDF[acsDF.Count != "Skip"]
ggplot(data = acsDF) + geom_point(aes(x = "Count", y= "LogFreq"))

In [61]:
# M lineage
vcfM = cyvcf2.VCF("/home/jobste/Data/Chr16_AllInfo_AC_MLineage.vcf.gz")
num = 0
acs = []
for variant in vcfM:
    if variant.INFO["Major"] == ancestralList[num]:
        acs.append(variant.INFO["AC"])
    else:
        if ancestralList[num] == 0:
            acs.append("Skip")
        else:
            acs.append(variant.INFO["AN"] - variant.INFO["AC"])
    num += 1

from collections import Counter
acsDF = pd.DataFrame.from_dict(Counter(acs), orient = "index")
acsDF.columns = ["Freq"]
acsDF.loc[:, "Count"] = acsDF.index
acsDF.loc[:, "LogFreq"] = np.log(acsDF.Freq)
acsDF = acsDF[acsDF.Count != "Skip"]

In [48]:
ggplot(data = acsDF) + geom_point(aes(x = "Count", y= "LogFreq"))

In [52]:
acs = [x for x,y in zip(acs, ancestralTypeList) if not y in ["MajorOutgroup_Minor"]] #, "MajorOutgroup_Major", "0", "Minor"
from collections import Counter
acsDF = pd.DataFrame.from_dict(Counter(acs), orient = "index")
acsDF.columns = ["Freq"]
acsDF.loc[:, "Count"] = acsDF.index
acsDF.loc[:, "LogFreq"] = np.log(acsDF.Freq)
acsDF = acsDF[acsDF.Count != "Skip"]
ggplot(data = acsDF) + geom_point(aes(x = "Count", y= "LogFreq"))

In [62]:
acs = [x for x,y in zip(acs, ancestralTypeList) if y == "Major"] #, "MajorOutgroup_Major", "0", "Minor"
from collections import Counter
acsDF = pd.DataFrame.from_dict(Counter(acs), orient = "index")
acsDF.columns = ["Freq"]
acsDF.loc[:, "Count"] = acsDF.index
acsDF.loc[:, "LogFreq"] = np.log(acsDF.Freq)
acsDF = acsDF[acsDF.Count != "Skip"]
ggplot(data = acsDF) + geom_point(aes(x = "Count", y= "LogFreq"))

In [59]:
# C lineage
vcfC = cyvcf2.VCF("/home/jobste/Data/Chr16_AllInfo_AC_CLineage.vcf.gz")
num = 0
acs = []
for variant in vcfC:
    if variant.INFO["Major"] == ancestralList[num]:
        acs.append(variant.INFO["AC"])
    else:
        if ancestralList[num] == 0:
            acs.append("Skip")
        else:
            acs.append(variant.INFO["AN"] - variant.INFO["AC"])
    num += 1

from collections import Counter
acsDF = pd.DataFrame.from_dict(Counter(acs), orient = "index")
acsDF.columns = ["Freq"]
acsDF.loc[:, "Count"] = acsDF.index
acsDF.loc[:, "LogFreq"] = np.log(acsDF.Freq)
acsDF = acsDF[acsDF.Count != "Skip"]

ggplot(data = acsDF) + geom_point(aes(x = "Count", y= "LogFreq"))

In [58]:
acs = [x for x,y in zip(acs, ancestralTypeList) if not y in ["MajorOutgroup_Minor"]] #, "MajorOutgroup_Major", "0", "Minor"
from collections import Counter
acsDF = pd.DataFrame.from_dict(Counter(acs), orient = "index")
acsDF.columns = ["Freq"]
acsDF.loc[:, "Count"] = acsDF.index
acsDF.loc[:, "LogFreq"] = np.log(acsDF.Freq)
acsDF = acsDF[acsDF.Count != "Skip"]
ggplot(data = acsDF) + geom_point(aes(x = "Count", y= "LogFreq"))

In [60]:
acs = [x for x,y in zip(acs, ancestralTypeList) if y == "Major"] #, "MajorOutgroup_Major", "0", "Minor"
from collections import Counter
acsDF = pd.DataFrame.from_dict(Counter(acs), orient = "index")
acsDF.columns = ["Freq"]
acsDF.loc[:, "Count"] = acsDF.index
acsDF.loc[:, "LogFreq"] = np.log(acsDF.Freq)
acsDF = acsDF[acsDF.Count != "Skip"]
ggplot(data = acsDF) + geom_point(aes(x = "Count", y= "LogFreq"))